# Reproducing the paper's $L^p$ and $W^{1,p}$ experiments (excluding $W^{2,p}$)

This notebook focuses on the same two norm families highlighted in the paper: certified
$L^p$ and $W^{1,p}$ bounds via adaptive interval refinement.

- Included: $L^p$, $W^{1,p}$
- Excluded on purpose: $W^{2,p}$

The examples below are fully reproducible from this repository and use `intervalnets`' current
`model.lpnorm(...)` and `model.sobolev_norm(...)` implementations.

In [ ]:
import sys
from pathlib import Path

# Add src/ to sys.path when running from a repository checkout.
repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import math
import random
import statistics
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from intervalnets import IntervalTensor, enable_interval_eval

enable_interval_eval()

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Using deterministic seed: {SEED}")

## Helper utilities

In [ ]:
def make_mlp(input_dim: int = 2, width: int = 32, depth: int = 3) -> nn.Sequential:
    layers = []
    in_features = input_dim
    for _ in range(depth):
        layers.append(nn.Linear(in_features, width))
        layers.append(nn.Tanh())
        in_features = width
    layers.append(nn.Linear(in_features, 1))
    model = nn.Sequential(*layers)
    # Small weights keep outputs/gradients numerically stable while still nontrivial.
    with torch.no_grad():
        for p in model.parameters():
            p.mul_(0.5)
    return model


def interval_width(itv):
    return float(itv.upper - itv.lower)


def mc_lp_estimate(model: nn.Module, dim: int, p: float, n: int = 200_000) -> float:
    with torch.no_grad():
        x = torch.rand(n, dim) * 2.0 - 1.0  # [-1,1]^dim
        y = model(x).squeeze(-1)
        vol = 2.0 ** dim
        integral = vol * torch.mean(torch.abs(y) ** p).item()
        return integral ** (1.0 / p)


def mc_w1p_estimate(model: nn.Module, dim: int, p: float, n: int = 60_000) -> float:
    # Empirical (not certified) estimator for comparison to certified bounds.
    x = torch.rand(n, dim, requires_grad=True) * 2.0 - 1.0
    y = model(x).squeeze(-1)
    grad = torch.autograd.grad(y.sum(), x, create_graph=False)[0]

    grad_norm = torch.linalg.vector_norm(grad, ord=2, dim=-1)
    integrand = torch.abs(y) ** p + grad_norm ** p
    vol = 2.0 ** dim
    integral = vol * torch.mean(integrand).item()
    return integral ** (1.0 / p)

## Experiment 1 — $L^p$ interval tightening under adaptive refinement

The key behavior to reproduce is that increasing refinement iterations tightens certified bounds.

In [ ]:
model_lp = make_mlp(input_dim=2, width=32, depth=3)
domain_2d = IntervalTensor.from_bounds([-1.0, -1.0], [1.0, 1.0])

iterations = list(range(0, 11))
p_values = [1.0, 2.0, 4.0]

lp_bounds_by_p = {p: [] for p in p_values}
lp_widths_by_p = {p: [] for p in p_values}

for p in p_values:
    for it in iterations:
        b = model_lp.lpnorm(domain_2d, p=p, iterations=it)
        lp_bounds_by_p[p].append(b)
        lp_widths_by_p[p].append(interval_width(b))

fig, ax = plt.subplots(figsize=(7, 4))
for p in p_values:
    ax.plot(iterations, lp_widths_by_p[p], marker="o", label=f"p={p:g}")
ax.set_yscale("log")
ax.set_xlabel("refinement iterations")
ax.set_ylabel("certified interval width")
ax.set_title("Lp certified bound tightening")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

for p in p_values:
    print(f"p={p:g}: width(iter=0)={lp_widths_by_p[p][0]:.6e}, width(iter=10)={lp_widths_by_p[p][-1]:.6e}")
    assert lp_widths_by_p[p][-1] <= lp_widths_by_p[p][0] + 1e-12

## Experiment 2 — $L^p$ certification against Monte Carlo reference

In [ ]:
mc_n = 120_000
for p in [2.0, 4.0]:
    certified = model_lp.lpnorm(domain_2d, p=p, iterations=10)
    empirical = mc_lp_estimate(model_lp, dim=2, p=p, n=mc_n)
    print(f"Lp p={p:g}: certified={certified}, empirical≈{empirical:.6f}")
    assert certified.lower <= empirical <= certified.upper

## Experiment 3 — $W^{1,p}$ interval tightening under adaptive refinement

This uses `model.sobolev_norm(...)` and intentionally skips any $W^{2,p}$ computations.

In [ ]:
model_w1p = make_mlp(input_dim=2, width=24, depth=2)

w1p_bounds_by_p = {p: [] for p in p_values}
w1p_widths_by_p = {p: [] for p in p_values}

for p in p_values:
    for it in iterations:
        b = model_w1p.sobolev_norm(domain_2d, p=p, iterations=it)
        w1p_bounds_by_p[p].append(b)
        w1p_widths_by_p[p].append(interval_width(b))

fig, ax = plt.subplots(figsize=(7, 4))
for p in p_values:
    ax.plot(iterations, w1p_widths_by_p[p], marker="o", label=f"p={p:g}")
ax.set_yscale("log")
ax.set_xlabel("refinement iterations")
ax.set_ylabel("certified interval width")
ax.set_title("W1p certified bound tightening")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

for p in p_values:
    print(f"p={p:g}: width(iter=0)={w1p_widths_by_p[p][0]:.6e}, width(iter=10)={w1p_widths_by_p[p][-1]:.6e}")
    assert w1p_widths_by_p[p][-1] <= w1p_widths_by_p[p][0] + 1e-12

## Experiment 4 — $W^{1,p}$ certification against an empirical autograd estimate

In [ ]:
for p in [2.0, 4.0]:
    certified = model_w1p.sobolev_norm(domain_2d, p=p, iterations=10)
    empirical = mc_w1p_estimate(model_w1p, dim=2, p=p, n=40_000)
    print(f"W1p p={p:g}: certified={certified}, empirical≈{empirical:.6f}")
    assert certified.lower <= empirical <= certified.upper

## Notes

- This notebook is intentionally scoped to **$L^p$** and **$W^{1,p}$** only.
- No $W^{2,p}$ code path is used.
- The main reproducibility signal is certified interval width decay as refinement increases,
  plus containment of empirical estimates.